# 🧩 Modular Functions — Retail Rental Analytics System

> A curated reference of the most important, reusable functions across the four core notebooks.  
> Each function is self-contained and can be dropped into any vertical or new notebook.
>
> **Notebooks covered:**
> - `01` · Data Generation
> - `02` · EDA
> - `03` · A/B Testing
> - `04` · Machine Learning

---

## 0 · Shared Setup

> Standard imports, `.env` loading, MySQL connection, and the `save()` helper used across all notebooks.  
> Copy this cell verbatim into any new notebook.

In [ ]:
import pandas as pd
import numpy as np
import os
from pathlib import Path
from datetime import datetime, timedelta

try:
    from sqlalchemy import create_engine, text
except Exception:
    create_engine = None
    text = None

try:
    from dotenv import load_dotenv
except Exception:
    def load_dotenv(): return None

load_dotenv()
np.random.seed(42)

DATA_DIR    = "../data/generated_data"
TABLEAU_DIR = "../data/tableau"
FIGURES_DIR = "../figures"
SQL_DIR     = "../data/sql"

for d in [DATA_DIR, TABLEAU_DIR, FIGURES_DIR, SQL_DIR]:
    os.makedirs(d, exist_ok=True)

# MySQL connection — graceful fallback to CSV-only if credentials not found
engine = None
if create_engine is not None and all(os.getenv(k) for k in ["DB_USER", "DB_PASSWORD", "DB_HOST", "DB_NAME"]):
    try:
        engine = create_engine(
            f"mysql+pymysql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
            f"@{os.getenv('DB_HOST')}/{os.getenv('DB_NAME')}",
            echo=False
        )
        with engine.begin() as conn:
            conn.execute(text("SELECT 1"))
        print("MySQL connection OK.")
    except Exception as e:
        print(f"MySQL unavailable. CSV-only mode. Reason: {e}")
        engine = None
else:
    print("MySQL credentials not found. CSV-only mode.")

def save(name, df):
    """Write DataFrame to CSV and optionally to MySQL in one call."""
    df.to_csv(f"{DATA_DIR}/{name}.csv", index=False)
    if engine is not None:
        df.to_sql(name, engine, if_exists="replace", index=False)
    print(f"  {name}: {len(df):,} rows")

---
## 1 · Data Generation

> Functions from `01_data_generation.ipynb` and the `01B_*` vertical generators.  
> These handle product listing, pricing rules, customer construction, rental simulation, and revenue comparison.

### 1.1 · `random_listed_date()`

Generates a realistic product listing date with a weighted year distribution.  
More recent years are more likely — simulating a growing inventory over time.  
The 2024 window is capped at mid-year so products have time to age into eligibility.

In [ ]:
def random_listed_date():
    """Return a random listing date, weighted toward more recent years."""
    year = np.random.choice([2020, 2021, 2022, 2023, 2024], p=[0.02, 0.08, 0.22, 0.36, 0.32])
    if year == 2024:
        return datetime(2024, 1, 1) + timedelta(days=int(np.random.uniform(0, 181)))
    return datetime(year, 1, 1) + timedelta(days=int(np.random.uniform(0, 365)))

### 1.2 · `sample_retail_price(category_id, price_bands_by_cat)`

Samples a retail price for a product from its category's price band distribution.  
Each band is a tuple of `(low, high, probability_weight)`.  
This ensures realistic price distributions without uniform randomness — expensive items are rarer.

In [ ]:
def sample_retail_price(category_id, price_bands_by_cat):
    """
    Sample a retail price from a category's price band distribution.
    price_bands_by_cat: dict of {category_id: [(low, high, prob), ...]}
    """
    bands = price_bands_by_cat[category_id]
    probs = [b[2] for b in bands]
    idx   = np.random.choice(range(len(bands)), p=probs)
    low, high, _ = bands[idx]
    return round(np.random.uniform(low, high), 2)

### 1.3 · `sample_brand(cat_id, brands_by_cat)`

Picks a brand name for a product using category-specific weighted probabilities.  
Market leaders get higher weights — mimics real shelf share without manual assignment.

In [ ]:
def sample_brand(cat_id, brands_by_cat):
    """
    Sample a brand from a category's weighted brand list.
    brands_by_cat: dict of {category_id: [(brand_name, probability), ...]}
    """
    brand_weights = brands_by_cat[cat_id]
    brands = [b[0] for b in brand_weights]
    probs  = [b[1] for b in brand_weights]
    return np.random.choice(brands, p=probs)

### 1.4 · `choose_duration(cat_id, demand, month)`

Selects a rental duration in days based on category type and the month of rental.  
Season-aware: winter months pull longer for outerwear; peak months pull longer for dresses.  
This is what creates the realistic duration distributions visible in the EDA charts.

In [ ]:
def choose_duration(cat_id, demand, month):
    """
    Return a rental duration in days, informed by category and season.
    cat_id 6 = Occasion Wear (short), cat_id 7 = Outerwear (seasonal).
    Adapt category IDs and distributions to your vertical.
    """
    if cat_id == 6:   # Occasion Wear: short event hires
        return int(np.random.choice([2, 3, 4, 5, 7], p=[0.20, 0.30, 0.25, 0.15, 0.10]))
    elif cat_id == 7: # Outerwear: seasonal
        dur = int(np.random.choice([60, 90, 120, 150, 180], p=[0.15, 0.28, 0.32, 0.15, 0.10]))
        if month in (11, 12, 1, 2):  # Winter pulls longer
            dur = int(np.random.choice([90, 120, 150, 180], p=[0.20, 0.35, 0.28, 0.17]))
        return dur
    elif cat_id in [4, 5]:  # Activewear, Nursing: weekly rotation
        return int(np.random.choice([7, 14, 21, 28], p=[0.30, 0.38, 0.22, 0.10]))
    else:  # Dresses, Tops, Jeans: trimester cycles
        dur = int(np.random.choice([21, 28, 42, 56, 84, 90], p=[0.12, 0.25, 0.28, 0.20, 0.10, 0.05]))
        if month in (2, 3, 8, 9):  # Peak pregnancy months pull longer
            dur = int(np.random.choice([28, 42, 56, 84], p=[0.20, 0.35, 0.30, 0.15]))
        return dur

### 1.5 · `choose_rental_count(cat_id, demand, days_available)`

Determines how many times a product is rented during its programme window.  
Capped by the number of days available — prevents physically impossible rental counts.  
High-demand, short-duration categories (Occasion Wear) turn over much faster than Outerwear.

In [ ]:
def choose_rental_count(cat_id, demand, days_available):
    """
    Return the number of rentals for a product during its programme window.
    days_available: total days the product is eligible for rental.
    """
    max_possible = max(1, days_available // 14)  # min 14-day turnaround assumed
    if cat_id == 6:   # Occasion: high turnover, short hires
        base = np.random.choice([5, 6, 7, 8, 9], p=[0.12, 0.22, 0.30, 0.24, 0.12])
    elif cat_id == 7: # Outerwear: 1–2 rentals per season
        base = np.random.choice([1, 2, 3], p=[0.40, 0.45, 0.15])
    elif demand == "high":
        base = np.random.choice([3, 4, 5, 6], p=[0.20, 0.32, 0.30, 0.18])
    else:  # medium
        base = np.random.choice([2, 3, 4, 5], p=[0.25, 0.38, 0.25, 0.12])
    return min(int(base), max_possible)

### 1.6 · `choose_pricing_model(cat_id, price)`

Selects a pricing model (`flat_rate` vs `pct_of_retail`) based on category and price tier.  
Higher-value items skew toward `pct_of_retail` — a €500 item earns more as a % of retail than a flat rate.  
This is what creates the unequal A/B groups: pricing model is assigned *before* the A/B label.

In [ ]:
def choose_pricing_model(cat_id, price):
    """
    Return 'flat_rate' or 'pct_of_retail' based on category and price point.
    Occasion Wear (high price, value-sensitive) skews pct_of_retail.
    Budget categories skew flat_rate.
    """
    if cat_id == 6:  # Occasion Wear: premium pricing by item value
        return np.random.choice(["pct_of_retail", "flat_rate"], p=[0.72, 0.28])
    elif price >= 500:
        return np.random.choice(["pct_of_retail", "flat_rate"], p=[0.65, 0.35])
    elif price < 200:
        return np.random.choice(["flat_rate", "pct_of_retail"], p=[0.72, 0.28])
    else:
        return np.random.choice(["flat_rate", "pct_of_retail"], p=[0.55, 0.45])

### 1.7 · `get_discount(months_unsold, dep_class)`

Returns the markdown discount percentage for a product based on how long it has been unsold and its depreciation class.  
Calibrated against real retail data: `slow` categories (Occasion Wear, Outerwear) hold value longer; `fast` categories (Activewear, Nursing) are discounted aggressively.  
This function is the baseline that every rental revenue comparison is measured against.

In [ ]:
def get_discount(months_unsold, dep_class):
    """
    Return the markdown discount rate for a product.
    dep_class: 'slow', 'standard', or 'fast'
    Tiers are (months_threshold, discount_pct) pairs.
    Calibrate tiers to your vertical's real resale data.
    """
    tiers = {
        "slow":     [(12, 0.20), (18, 0.35), (24, 0.48), (999, 0.58)],
        "standard": [(12, 0.30), (18, 0.45), (24, 0.58), (999, 0.68)],
        "fast":     [(12, 0.40), (18, 0.55), (24, 0.65), (999, 0.72)],
    }
    for threshold, pct in tiers[dep_class]:
        if months_unsold <= threshold:
            return pct
    return tiers[dep_class][-1][1]

### 1.8 · `next_customer(month, cat_id)` — Segment-Aware Customer Selector

Pulls from a pre-shuffled customer pool, filtering by segment rules and applying monthly demand boosts.  
Ensures postpartum customers only rent from nursing categories, and vice versa.  
The boost mechanism is what creates realistic seasonal rental patterns in the data.

In [ ]:
# Requires: customer_pool (np.array), customer_segment_map (dict), SEG_MONTH_BOOST (dict)
# These are built in the customer and customer-mix cells of the data generation notebook.

pool_idx = 0

def next_customer(month=None, cat_id=None):
    """
    Return a customer_id from the pool, optionally filtered by segment rules.
    - Postpartum customers exclusively rent cat 5 (Nursing)
    - Non-postpartum customers are excluded from cat 5
    - Monthly boosts increase selection probability for seasonally active segments
    """
    global pool_idx
    for _ in range(10):
        if pool_idx >= len(customer_pool):
            pool_idx = 0
            np.random.shuffle(customer_pool)
        cid = int(customer_pool[pool_idx]); pool_idx += 1
        if month is None and cat_id is None:
            return cid
        seg = customer_segment_map.get(cid, "second_trimester")
        if seg == "postpartum" and cat_id is not None and cat_id != 5:
            continue
        if seg != "postpartum" and cat_id == 5:
            continue
        boost = SEG_MONTH_BOOST.get(seg, {}).get(month, 1.0) if month else 1.0
        if np.random.random() < boost / 1.28:
            return cid
    # Fallback: return next available without boost check
    if pool_idx >= len(customer_pool):
        pool_idx = 0
    cid = int(customer_pool[pool_idx]); pool_idx += 1
    return cid

---
## 2 · EDA

> Functions from `02_eda.ipynb`.  
> These are charting and summarisation helpers used to explore the rental dataset.

### 2.1 · `plot_inventory_aging(products_df, figures_dir)`

Plots the distribution of days on shelf by category.  
The key visual for demonstrating the 365-day threshold: shows clearly which products crossed the eligibility line and how many are in the clearance zone.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

def plot_inventory_aging(products_df, figures_dir):
    """
    Plot days-on-shelf distribution per category with the 365-day eligibility threshold marked.
    Saves to figures_dir/inventory_aging.png
    """
    products_df = products_df.copy()
    products_df["listed_date"] = pd.to_datetime(products_df["listed_date"])
    products_df["days_on_shelf"] = (pd.Timestamp("2024-12-31") - products_df["listed_date"]).dt.days

    fig, ax = plt.subplots(figsize=(12, 5))
    for cat, grp in products_df.groupby("category_id"):
        ax.hist(grp["days_on_shelf"], bins=40, alpha=0.5, label=f"Cat {cat}")
    ax.axvline(365, color="red", linestyle="--", linewidth=1.5, label="365-day threshold")
    ax.set_xlabel("Days on Shelf")
    ax.set_ylabel("Product Count")
    ax.set_title("Inventory Aging — Days on Shelf by Category")
    ax.legend(fontsize=8, ncol=3)
    plt.tight_layout()
    plt.savefig(f"{figures_dir}/inventory_aging.png", dpi=150)
    plt.show()
    print(f"Saved: {figures_dir}/inventory_aging.png")

### 2.2 · `repeat_renter_summary(rentals_df, customers_df)`

Identifies repeat renters and computes their share of total revenue.  
The key EDA finding: ~30% of customers drive ~50% of revenue.  
This function reproduces that summary table from any rentals + customers DataFrame pair.

In [ ]:
def repeat_renter_summary(rentals_df, customers_df, repeat_threshold=2):
    """
    Return a summary dict: share of repeat renters and their revenue contribution.
    repeat_threshold: minimum rentals to count as a repeat renter (default 2).
    """
    rental_counts = rentals_df.groupby("customer_id")["rental_id"].count().reset_index()
    rental_counts.columns = ["customer_id", "rental_count"]

    repeat_ids = rental_counts[rental_counts["rental_count"] >= repeat_threshold]["customer_id"]
    repeat_pct = len(repeat_ids) / len(rental_counts)

    total_rev  = rentals_df["total_rental_revenue"].sum()
    repeat_rev = rentals_df[rentals_df["customer_id"].isin(repeat_ids)]["total_rental_revenue"].sum()
    repeat_rev_pct = repeat_rev / total_rev if total_rev > 0 else 0

    summary = {
        "total_customers_who_rented": len(rental_counts),
        "repeat_renters":             len(repeat_ids),
        "repeat_renter_pct":          round(repeat_pct, 4),
        "total_revenue":              round(total_rev, 2),
        "repeat_renter_revenue":      round(repeat_rev, 2),
        "repeat_renter_revenue_pct":  round(repeat_rev_pct, 4),
    }
    for k, v in summary.items():
        print(f"  {k}: {v}")
    return summary

### 2.3 · `win_rate_by_category(comparison_df, categories_df)`

Computes the rental win rate and median revenue ratio per category.  
The printout mirrors the summary at the end of every data generation notebook.  
Scope to programme categories only — always filter out non-programme rows before calling this.

In [ ]:
def win_rate_by_category(comparison_df, categories_df):
    """
    Print and return per-category win rate and median revenue ratio.
    comparison_df must be scoped to programme categories only (total_gross_rental_revenue > 0).
    """
    # IMPORTANT: scope to eligible programme products — excludes guaranteed losses
    comp = comparison_df[comparison_df["total_gross_rental_revenue"] > 0].copy()

    cat_names = categories_df.set_index("category_id")["category_name"].to_dict()
    prog_cats  = categories_df[categories_df["rental_programme"] == True]["category_id"].tolist()

    results = []
    for cid in prog_cats:
        cat_comp = comp[comp["product_id"].isin(
            comp[comp["product_id"] == cid]["product_id"] if False else
            comp.index  # placeholder — join with products on category_id in practice
        )]
        # In practice: filter comp by products[products['category_id'] == cid]['product_id']

    # Simpler: pass pre-merged DataFrame with category_id included
    if "category_id" in comp.columns:
        for cid in prog_cats:
            cat_comp = comp[comp["category_id"] == cid]
            if len(cat_comp) == 0:
                continue
            win  = cat_comp["is_rental_more_profitable"].mean()
            med  = cat_comp["rental_vs_discount_ratio"].median()
            name = cat_names.get(cid, f"Cat {cid}")
            print(f"  {name:<40} win={win:.1%}  median={med:.2f}x")
            results.append({"category_id": cid, "category_name": name, "win_rate": win, "median_ratio": med})
    return pd.DataFrame(results)

---
## 3 · A/B Testing

> Functions from `03_ab_testing.ipynb`.  
> Statistical tests used to determine whether observed differences are real or noise.

### 3.1 · `run_welch_ttest(group_a, group_b, label)`

Runs a two-sample Welch's t-test comparing revenue between experiment groups.  
Welch's is used here — not Student's — because the groups are unequal in size by design (pricing model correlates with price tier).  
Returns the t-statistic, p-value, and a plain-English verdict.

In [ ]:
from scipy import stats

def run_welch_ttest(group_a, group_b, label="", alpha=0.05):
    """
    Run a two-sample Welch's t-test (unequal variance, unequal sample size).
    group_a, group_b: array-like of revenue values
    Returns dict with t-stat, p-value, and significance verdict.
    """
    t_stat, p_val = stats.ttest_ind(group_a, group_b, equal_var=False)
    significant = p_val < alpha

    print(f"{'='*50}")
    print(f"Welch's t-test{' — ' + label if label else ''}")
    print(f"  Group A:  n={len(group_a):,}  mean={np.mean(group_a):.2f}  std={np.std(group_a):.2f}")
    print(f"  Group B:  n={len(group_b):,}  mean={np.mean(group_b):.2f}  std={np.std(group_b):.2f}")
    print(f"  t-stat:   {t_stat:.4f}")
    print(f"  p-value:  {p_val:.4f}")
    print(f"  Result:   {'SIGNIFICANT ✅' if significant else 'NOT significant ❌'} (α={alpha})")
    print(f"{'='*50}")

    return {"t_stat": t_stat, "p_value": p_val, "significant": significant,
            "mean_a": np.mean(group_a), "mean_b": np.mean(group_b)}

### 3.2 · `bootstrap_confidence_interval(data, stat_fn, n_iter, ci)`

Computes a bootstrap confidence interval for any statistic function.  
Used in the A/B notebook to build CIs on win rate and median revenue ratio — two statistics where analytical CIs are unreliable.  
Pass any function as `stat_fn`: `np.mean`, `np.median`, or a custom lambda.

In [ ]:
def bootstrap_confidence_interval(data, stat_fn=np.mean, n_iter=10_000, ci=0.95):
    """
    Return a bootstrap confidence interval for a statistic.
    data:    array-like of observed values
    stat_fn: statistic to compute on each resample (default: np.mean)
    n_iter:  number of bootstrap resamples (default: 10,000)
    ci:      confidence level (default: 0.95)
    """
    data = np.array(data)
    boot_stats = [
        stat_fn(np.random.choice(data, size=len(data), replace=True))
        for _ in range(n_iter)
    ]
    lower = np.percentile(boot_stats, (1 - ci) / 2 * 100)
    upper = np.percentile(boot_stats, (1 + ci) / 2 * 100)
    observed = stat_fn(data)

    print(f"  Observed {stat_fn.__name__}: {observed:.4f}")
    print(f"  {int(ci*100)}% Bootstrap CI: [{lower:.4f}, {upper:.4f}]")
    return {"observed": observed, "lower": lower, "upper": upper, "ci": ci}

### 3.3 · `run_one_sample_ttest(sample, null_mean, label)`

Tests whether the observed win rate is statistically different from the null hypothesis (rental = markdown, i.e. ratio = 1.0).  
The core test of the thesis: is the rental win rate significantly above 50%, or could it be explained by chance?

In [ ]:
def run_one_sample_ttest(sample, null_mean=1.0, label="", alpha=0.05):
    """
    One-sample t-test against a null hypothesis mean.
    Default null: ratio = 1.0 (rental revenue = markdown revenue).
    sample: array of rental_vs_discount_ratio values
    """
    t_stat, p_val = stats.ttest_1samp(sample, popmean=null_mean)
    significant = p_val < alpha

    print(f"{'='*50}")
    print(f"One-sample t-test{' — ' + label if label else ''}")
    print(f"  H₀: mean ratio = {null_mean}  (rental = markdown)")
    print(f"  n={len(sample):,}  observed mean={np.mean(sample):.4f}  std={np.std(sample):.4f}")
    print(f"  t-stat:  {t_stat:.4f}")
    print(f"  p-value: {p_val:.4e}")
    print(f"  Result:  {'REJECT H₀ ✅' if significant else 'FAIL TO REJECT H₀ ❌'} (α={alpha})")
    print(f"{'='*50}")

    return {"t_stat": t_stat, "p_value": p_val, "significant": significant,
            "observed_mean": np.mean(sample), "null_mean": null_mean}

---
## 4 · Machine Learning

> Functions from `04_machine_learning.ipynb`.  
> Model evaluation, sensitivity analysis, and the Monte Carlo simulation.

### 4.1 · `evaluate_classifier(model, X_test, y_test, label)`

Prints a full classification report plus AUC-ROC for any sklearn-compatible classifier.  
Used for the Random Forest (Model 1) and Logistic Regression (Model 3).  
Returns the AUC score for easy comparison across models.

In [ ]:
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

def evaluate_classifier(model, X_test, y_test, label=""):
    """
    Print classification report and AUC-ROC for a fitted classifier.
    Returns AUC score.
    """
    y_pred  = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else y_pred
    auc     = roc_auc_score(y_test, y_proba)

    print(f"{'='*50}")
    print(f"Classifier Evaluation{' — ' + label if label else ''}")
    print(classification_report(y_test, y_pred))
    print(f"  AUC-ROC: {auc:.4f}")
    print(f"  Confusion matrix:\n{confusion_matrix(y_test, y_pred)}")
    print(f"{'='*50}")
    return auc

### 4.2 · `sensitivity_analysis(comparison_df, scenarios)`

Re-runs the win rate calculation under multiple cost scenarios (optimistic / realistic / pessimistic).  
Each scenario reapplies a different operational cost multiplier to gross rental revenue.  
This is Section 4a of the ML notebook — the precursor to Monte Carlo.

In [ ]:
def sensitivity_analysis(comparison_df, scenarios=None):
    """
    Compute win rate and median ratio under different operational cost assumptions.
    scenarios: list of (label, op_cost_pct) tuples. Default: optimistic/realistic/pessimistic.
    comparison_df must include: total_gross_rental_revenue, hypothetical_discount_price
    """
    if scenarios is None:
        scenarios = [
            ("Optimistic",   0.15),
            ("Realistic",    0.22),
            ("Pessimistic",  0.35),
        ]

    # Scope to programme products only
    comp = comparison_df[comparison_df["total_gross_rental_revenue"] > 0].copy()

    print(f"{'Scenario':<16} {'Op Cost':>8} {'Win Rate':>10} {'Median Ratio':>14} {'n':>6}")
    print("-" * 58)
    results = []
    for label, op_pct in scenarios:
        net = comp["total_gross_rental_revenue"] * (1 - op_pct)
        ratio = net / comp["hypothetical_discount_price"]
        win_rate    = (ratio > 1.0).mean()
        median_ratio = ratio.median()
        print(f"  {label:<14} {op_pct:>7.0%} {win_rate:>10.1%} {median_ratio:>13.2f}x {len(comp):>6}")
        results.append({"scenario": label, "op_cost_pct": op_pct,
                         "win_rate": win_rate, "median_ratio": median_ratio})
    return pd.DataFrame(results)

### 4.3 · `run_monte_carlo(comparison_df, n_simulations)`

The headline stress test. Runs `n_simulations` iterations, randomising operational costs, no-return rates, and damage write-offs simultaneously using triangular distributions.

**Critical:** scope `comparison_df` to programme categories only (`total_gross_rental_revenue > 0`) before calling — otherwise non-eligible products are counted as guaranteed losses in every simulation.  
That was the bug that changed the results from 59.6% → 100% win rate.

In [ ]:
def run_monte_carlo(comparison_df, n_simulations=10_000):
    """
    Monte Carlo simulation: vary all uncertain inputs simultaneously.
    Inputs randomised per simulation:
      - Operational cost: triangular(15%, 22%, 35%)
      - No-return rate:   triangular(4%,  6%,  10%)
      - Damage write-off: triangular(2%,  4%,   8%)

    IMPORTANT: pass comparison_df scoped to rental_programme products only.
    i.e. comparison_df = comparison_df[comparison_df['total_gross_rental_revenue'] > 0].copy()

    Returns DataFrame of per-simulation results.
    """
    # Scope guard — always filter here as a safety net
    comp = comparison_df[comparison_df["total_gross_rental_revenue"] > 0].copy()

    gross_rev    = comp["total_gross_rental_revenue"].values
    disc_price   = comp["hypothetical_discount_price"].values

    sim_results = []
    for _ in range(n_simulations):
        op_cost_pct   = np.random.triangular(0.15, 0.22, 0.35)
        no_return_pct = np.random.triangular(0.04, 0.06, 0.10)
        damage_pct    = np.random.triangular(0.02, 0.04, 0.08)

        # Apply all three cost factors
        adjusted_rev  = gross_rev * (1 - op_cost_pct) * (1 - no_return_pct) * (1 - damage_pct)
        ratios        = adjusted_rev / disc_price
        win_rate      = (ratios > 1.0).mean()
        median_ratio  = np.median(ratios)

        sim_results.append({"win_rate": win_rate, "median_ratio": median_ratio,
                             "op_cost_pct": op_cost_pct, "no_return_pct": no_return_pct,
                             "damage_pct": damage_pct})

    mc = pd.DataFrame(sim_results)
    rental_won = (mc["win_rate"] > 0.5).mean()

    print(f"{'='*50}")
    print(f"Monte Carlo — {n_simulations:,} simulations")
    print(f"  Rental won in:  {rental_won:.1%} of simulations")
    print(f"  Win rate range: {mc['win_rate'].quantile(0.05):.1%} – {mc['win_rate'].quantile(0.95):.1%} (5th–95th pct)")
    print(f"  Median ratio:   {mc['median_ratio'].median():.2f}x")
    print(f"  Ratio range:    {mc['median_ratio'].quantile(0.05):.2f}x – {mc['median_ratio'].quantile(0.95):.2f}x")
    print(f"{'='*50}")
    return mc

### 4.4 · `plot_feature_importance(model, feature_names, top_n, figures_dir)`

Plots feature importance from a fitted Random Forest or any tree-based model.  
Used for Model 1 (rental profitability classifier) to show which product attributes most predict rental success.  
Replaces SHAP — simpler, more explainable at bootcamp level, and sufficient for the business story.

In [ ]:
def plot_feature_importance(model, feature_names, top_n=15, figures_dir="../figures"):
    """
    Bar chart of top_n most important features from a tree-based model.
    model: fitted sklearn estimator with .feature_importances_
    feature_names: list of feature names matching X columns
    """
    importances = pd.Series(model.feature_importances_, index=feature_names)
    top = importances.nlargest(top_n).sort_values()

    fig, ax = plt.subplots(figsize=(8, top_n * 0.4 + 1))
    top.plot(kind="barh", ax=ax, color="steelblue", edgecolor="white")
    ax.set_xlabel("Feature Importance")
    ax.set_title(f"Top {top_n} Feature Importances")
    plt.tight_layout()
    plt.savefig(f"{figures_dir}/feature_importance.png", dpi=150)
    plt.show()
    print(f"Saved: {figures_dir}/feature_importance.png")
    return top

---

## 5 · Tableau Flat File Builder

> From `01_data_generation.ipynb` — the `.merge()` chain that pre-joins all tables into a single flat file for Tableau.

### 5.1 · `build_tableau_flat_file(...)`

Joins all 6 tables into one denormalised CSV: one row per rental, every dimension already attached.  
This is the Python equivalent of a full SQL JOIN done upstream — Tableau never needs to join anything itself.  
When Power BI replaced Tableau, this was dropped in favour of a live MySQL connection. Kept here for reference.

In [ ]:
def build_tableau_flat_file(rentals, products, categories, customers, pricing, returns, comparison,
                             tableau_dir="../data/tableau"):
    """
    Build a fully denormalised flat file for Tableau (or any tool that doesn't support live joins).
    Returns the merged DataFrame and saves it to tableau_dir/rental_analysis_full.csv

    Column count will vary by vertical — expect 40–50 columns.
    """
    flat = (
        rentals
        .merge(products[["product_id", "category_id", "product_name", "brand",
                          "original_retail_price", "condition_grade", "listed_date",
                          "rental_eligible_date"]],
               on="product_id", how="left")
        .merge(categories[["category_id", "category_name", "depreciation_class",
                            "rental_demand_tier"]],
               on="category_id", how="left")
        .merge(customers[["customer_id", "city", "country", "customer_segment"]],
               on="customer_id", how="left")
        .merge(pricing[["rule_id", "pricing_model", "duration_model", "experiment_group"]],
               left_on="pricing_rule_id", right_on="rule_id", how="left")
        .merge(returns[["rental_id", "condition_on_return", "damage_fee"]],
               on="rental_id", how="left")
        .merge(comparison[["product_id", "hypothetical_discount_price",
                            "rental_vs_discount_ratio", "is_rental_more_profitable"]],
               on="product_id", how="left")
    )

    out_path = f"{tableau_dir}/rental_analysis_full.csv"
    flat.to_csv(out_path, index=False)
    print(f"  Flat file: {len(flat):,} rows × {len(flat.columns)} columns → {out_path}")
    return flat

---

> **Note:** These functions are vertical-agnostic. The data generation functions adapt to any vertical by swapping category definitions, price bands, brand lists, and depreciation tiers.  
> The statistical and ML functions are completely portable — they operate on DataFrames, not on vertical-specific logic.
>
> *Built as part of the Ironhack Data Analytics Final Project — April 2026*